### Load Checkpoint

In [1]:
from pathlib import Path
import torch

project = Path.cwd().parent

checkpoint = torch.load(
    project /
    "models" /
    "rgcn_complete_checkpoint.pt",
    map_location="cpu"
)

print(checkpoint.keys())

dict_keys(['model_state_dict', 'predictor_state_dict', 'node_embeddings', 'node2id', 'relation2id'])


### Recover Data

In [2]:
node_embeddings = checkpoint["node_embeddings"]

node2id = checkpoint["node2id"]

relation2id = checkpoint["relation2id"]

print(node_embeddings.shape)

torch.Size([7502, 128])


### Imports

In [3]:
import torch.nn as nn
import torch.nn.functional as F

### Build Transformer Layer

In [4]:
class ThreatTransformer(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
        num_heads=4,
        num_layers=2
    ):

        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        return self.transformer(x)

### Initialize Model

In [5]:
tx_model = ThreatTransformer(
    embedding_dim=128,
    num_heads=4,
    num_layers=2
)

tx_model

ThreatTransformer(
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)

### Prepare Embeddings
- Transformer expects: (batch, sequence, feature)

In [6]:
emb_input = node_embeddings.unsqueeze(0)

print(emb_input.shape)

torch.Size([1, 7502, 128])


### Initial Forward Pass
- Pass embeddings through transformer (with gradients enabled for training)

In [7]:
# Initial forward pass — gradients are ON so the transformer trains end-to-end
tx_embeddings = tx_model(emb_input)

print(tx_embeddings.shape)

torch.Size([1, 7502, 128])


### Remove Batch Dimension

In [8]:
tx_embeddings = tx_embeddings.squeeze(0)

print(tx_embeddings.shape)

torch.Size([7502, 128])


In [9]:
print(tx_embeddings.shape)

torch.Size([7502, 128])


--------------

### Tx Link Predictor

In [10]:
class TxLinkPredictor(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            embedding_dim * 2,
            128
        )

        self.fc2 = nn.Linear(
            128,
            64
        )

        self.fc3 = nn.Linear(
            64,
            1
        )

    def forward(
        self,
        src,
        dst
    ):

        x = torch.cat(
            [src, dst],
            dim=1
        )

        x = F.relu(
            self.fc1(x)
        )

        x = F.relu(
            self.fc2(x)
        )

        x = self.fc3(x)

        return x

### Initialize

In [11]:
tx_predictor = TxLinkPredictor()

tx_predictor

TxLinkPredictor(
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)

### Build Training Dataset
- as We already built: edge_index,edge_type

### Load edge_index and edge_type

In [12]:
import os

os.listdir(
    project / "data" / "ml_ready"
)

['edge_index.npy', 'edge_type.npy', 'node2id.pkl', 'relation2id.pkl']

In [13]:
import numpy as np

edge_index = torch.tensor(
    np.load(
        project /
        "data" /
        "ml_ready" /
        "edge_index.npy"
    ),
    dtype=torch.long
)

edge_type = torch.tensor(
    np.load(
        project /
        "data" /
        "ml_ready" /
        "edge_type.npy"
    ),
    dtype=torch.long
)

print(edge_index.shape)
print(edge_type.shape)

torch.Size([2, 15573])
torch.Size([15573])


In [14]:
num_nodes = node_embeddings.shape[0]

print(num_nodes)

7502


In [15]:
print(edge_index.shape)
print(edge_type.shape)
print(node_embeddings.shape)

torch.Size([2, 15573])
torch.Size([15573])
torch.Size([7502, 128])


---------------

### Build Positive Edges

In [16]:
src_nodes = edge_index[0]
dst_nodes = edge_index[1]

print(src_nodes.shape)
print(dst_nodes.shape)

torch.Size([15573])
torch.Size([15573])


### Build Negative Edges
- just like R-GCN

In [17]:
num_edges = edge_index.shape[1]
num_nodes = node_embeddings.shape[0]

negative_src = torch.randint(
    0,
    num_nodes,
    (num_edges,)
)

negative_dst = torch.randint(
    0,
    num_nodes,
    (num_edges,)
)

print(negative_src.shape)

torch.Size([15573])


### Create Training Samples

**Positive Label**

In [18]:
positive_labels = torch.ones(num_edges)

**negative label**

In [19]:
negative_labels = torch.zeros(num_edges)

### Combine Dataset

In [20]:
all_src = torch.cat(
    [src_nodes, negative_src]
)

all_dst = torch.cat(
    [dst_nodes, negative_dst]
)

all_labels = torch.cat(
    [positive_labels, negative_labels]
)

print(all_src.shape)
print(all_labels.shape)

torch.Size([31146])
torch.Size([31146])


### Train/Test Split

In [21]:
from sklearn.model_selection import train_test_split

indices = torch.arange(
    len(all_labels)
)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.3,
    random_state=42
)

print(len(train_idx))
print(len(test_idx))

21802
9344


### Optimizer

In [22]:
import torch.optim as optim

# Re-initialise both models so we start fresh
tx_model = ThreatTransformer(
    embedding_dim=128,
    num_heads=4,
    num_layers=2
)

tx_predictor = TxLinkPredictor()

# Joint optimizer — trains transformer + predictor together end-to-end
optimizer = optim.Adam(
    list(tx_model.parameters()) +
    list(tx_predictor.parameters()),
    lr=0.001
)

criterion = torch.nn.BCEWithLogitsLoss()



### Training Loop

In [23]:
epochs = 20

# node_embeddings from R-GCN checkpoint — used as input to transformer
# Shape: (num_nodes, 128)  →  unsqueeze to (1, num_nodes, 128) for transformer
emb_input = node_embeddings.unsqueeze(0)  # keep on cpu, no grad needed here

for epoch in range(epochs):

    tx_model.train()
    tx_predictor.train()

    # ── Forward pass through transformer (gradients flow) ──
    tx_out = tx_model(emb_input)          # (1, num_nodes, 128)
    tx_out = tx_out.squeeze(0)            # (num_nodes, 128)

    # ── Gather node embeddings for this batch ──
    src_embed = tx_out[
        all_src[train_idx]
    ]

    dst_embed = tx_out[
        all_dst[train_idx]
    ]

    labels = all_labels[
        train_idx
    ].unsqueeze(1)

    # ── Link prediction ──
    predictions = tx_predictor(
        src_embed,
        dst_embed
    )

    loss = criterion(
        predictions,
        labels
    )

    # ── Backprop through predictor AND transformer ──
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1} Loss: {loss.item():.4f}"
    )

Epoch 1 Loss: 0.6880
Epoch 2 Loss: 0.6320
Epoch 3 Loss: 0.5634
Epoch 4 Loss: 0.5028
Epoch 5 Loss: 0.4472
Epoch 6 Loss: 0.4015
Epoch 7 Loss: 0.3606
Epoch 8 Loss: 0.3277
Epoch 9 Loss: 0.2924
Epoch 10 Loss: 0.2674
Epoch 11 Loss: 0.2484
Epoch 12 Loss: 0.2420
Epoch 13 Loss: 0.2493
Epoch 14 Loss: 0.2276
Epoch 15 Loss: 0.2199
Epoch 16 Loss: 0.2181
Epoch 17 Loss: 0.1988
Epoch 18 Loss: 0.2079
Epoch 19 Loss: 0.1899
Epoch 20 Loss: 0.2031


-------------

### Evaluation

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

tx_model.eval()
tx_predictor.eval()

with torch.no_grad():

    # Re-run transformer with trained weights to get final embeddings
    tx_embeddings = tx_model(
        node_embeddings.unsqueeze(0)
    ).squeeze(0)

    src_embed = tx_embeddings[
        all_src[test_idx]
    ]

    dst_embed = tx_embeddings[
        all_dst[test_idx]
    ]

    logits = tx_predictor(
        src_embed,
        dst_embed
    )

    probs = torch.sigmoid(
        logits
    ).squeeze()

    preds = (
        probs > 0.5
    ).int()

    y_true = all_labels[
        test_idx
    ].numpy()

    y_pred = preds.numpy()

    y_prob = probs.numpy()

print(
    "Accuracy :",
    accuracy_score(y_true, y_pred)
)

print(
    "Precision:",
    precision_score(y_true, y_pred)
)

print(
    "Recall   :",
    recall_score(y_true, y_pred)
)

print(
    "F1 Score :",
    f1_score(y_true, y_pred)
)

print(
    "AUC      :",
    roc_auc_score(y_true, y_prob)
)

Accuracy : 0.9273330479452054
Precision: 0.887248322147651
Recall   : 0.9807121661721068
F1 Score : 0.9316420014094433
AUC      : 0.9716171951043617


### Save the Transformer model:

In [25]:
# Save transformer encoder weights
torch.save(
    tx_model.state_dict(),
    project / "models" / "tx_model.pth"
)

# Save link predictor weights
torch.save(
    tx_predictor.state_dict(),
    project / "models" / "tx_predictor.pth"
)

# Save full checkpoint with both models together
torch.save(
    {
        "tx_model_state_dict": tx_model.state_dict(),
        "tx_predictor_state_dict": tx_predictor.state_dict()
    },
    project / "models" / "tx_complete_checkpoint.pt"
)

print("Saved: tx_model.pth, tx_predictor.pth, tx_complete_checkpoint.pt")

Saved: tx_model.pth, tx_predictor.pth, tx_complete_checkpoint.pt


In [26]:
print(type(tx_model))

<class '__main__.ThreatTransformer'>


In [27]:
print(tx_model)

ThreatTransformer(
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)


### save metrics table

In [28]:
import pandas as pd
comparison_df = pd.DataFrame({
    "Model":[
        "GraphSAGE",
        "R-GCN",
        "ThreatGraphX-Tx"
    ],
    "Accuracy":[
        0.9019,
        0.9234,
        0.9003
    ],
    "F1":[
        0.9088,
        0.9275,
        0.9020
    ],
    "AUC":[
        0.9514,
        0.9717,
        0.9656
    ]
})

comparison_df

,Model,Accuracy,F1,AUC
0,GraphSAGE,0.9019,0.9088,0.9514
1,R-GCN,0.9234,0.9275,0.9717
2,ThreatGraphX-Tx,0.9003,0.9020,0.9656


In [29]:
comparison_df.to_csv(
    project / "results" / "model_comparison.csv",
    index=False
)

----------------

## Zero-Shot Threat Prediction

### Find All CVEs

In [30]:
cve_nodes = [
    node
    for node in node2id.keys()
    if str(node).startswith("CVE-")
]

print("Total CVEs:", len(cve_nodes))

Total CVEs: 2024


### Select Hidden CVEs

In [31]:
import random

random.seed(42)

hidden_cves = random.sample(
    cve_nodes,
    int(len(cve_nodes) * 0.20)
)

print(
    "Hidden CVEs:",
    len(hidden_cves)
)

Hidden CVEs: 404


### Convert to IDs

In [32]:
hidden_ids = set(
    node2id[cve]
    for cve in hidden_cves
)

print(
    len(hidden_ids)
)

404


### Identify Edges to Remove

In [33]:
hidden_edge_mask = []

for i in range(edge_index.shape[1]):

    src = edge_index[0, i].item()

    dst = edge_index[1, i].item()

    if (
        src in hidden_ids
        or
        dst in hidden_ids
    ):
        hidden_edge_mask.append(True)
    else:
        hidden_edge_mask.append(False)

### Create Mask Tensor

In [34]:
hidden_edge_mask = torch.tensor(
    hidden_edge_mask
)

print(
    hidden_edge_mask.shape
)

torch.Size([15573])


### Count Hidden Edges

In [35]:
print(
    "Hidden edges:",
    hidden_edge_mask.sum().item()
)

print(
    "Remaining edges:",
    (~hidden_edge_mask).sum().item()
)

Hidden edges: 921
Remaining edges: 14652


- hide ~20% of CVEs

### Create Train Graph

In [36]:
train_edge_index = edge_index[
    :,
    ~hidden_edge_mask
]

train_edge_type = edge_type[
    ~hidden_edge_mask
]

print(train_edge_index.shape)
print(train_edge_type.shape)

torch.Size([2, 14652])
torch.Size([14652])


### Create Zero-Shot Test Graph

In [37]:
zero_shot_edge_index = edge_index[
    :,
    hidden_edge_mask
]

zero_shot_edge_type = edge_type[
    hidden_edge_mask
]

print(zero_shot_edge_index.shape)
print(zero_shot_edge_type.shape)

torch.Size([2, 921])
torch.Size([921])


### Verify Split

In [38]:
print(
    train_edge_index.shape[1]
    +
    zero_shot_edge_index.shape[1]
)

15573


At this point we are not retraining R-GCN from scratch.

Because true TxGNN-style retraining would require rebuilding the graph and rerunning the whole GNN training pipeline on the reduced graph.

That's possible, but a much larger experiment.

For a thesis, we can first perform a practical zero-shot evaluation using the learned embeddings and predictor.

### Zero-Shot Evaluation Dataset

In [39]:
zs_src = zero_shot_edge_index[0]
zs_dst = zero_shot_edge_index[1]

print(len(zs_src))

921


### Create Negative Samples

In [40]:
num_zs = len(zs_src)

negative_src = torch.randint(
    0,
    tx_embeddings.shape[0],
    (num_zs,)
)

negative_dst = torch.randint(
    0,
    tx_embeddings.shape[0],
    (num_zs,)
)

print(len(negative_src))

921


### Build Zero-Shot Test Set

In [41]:
zs_all_src = torch.cat(
    [zs_src, negative_src]
)

zs_all_dst = torch.cat(
    [zs_dst, negative_dst]
)

zs_labels = torch.cat(
    [
        torch.ones(num_zs),
        torch.zeros(num_zs)
    ]
)

print(zs_all_src.shape)
print(zs_labels.shape)

torch.Size([1842])
torch.Size([1842])


In [42]:
print(train_edge_index.shape)
print(zero_shot_edge_index.shape)

print(zs_all_src.shape)
print(zs_labels.shape)

torch.Size([2, 14652])
torch.Size([2, 921])
torch.Size([1842])
torch.Size([1842])


### Run Zero-Shot Predictions

In [43]:
tx_model.eval()
tx_predictor.eval()

with torch.no_grad():

    # Generate embeddings using the trained transformer
    tx_embeddings = tx_model(
        node_embeddings.unsqueeze(0)
    ).squeeze(0)

    src_embed = tx_embeddings[
        zs_all_src
    ]

    dst_embed = tx_embeddings[
        zs_all_dst
    ]

    logits = tx_predictor(
        src_embed,
        dst_embed
    )

    probs = torch.sigmoid(
        logits
    ).squeeze()

    preds = (
        probs > 0.5
    ).int()

### Compute Metrics

In [44]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_true = zs_labels.numpy()
y_pred = preds.numpy()
y_prob = probs.numpy()

print(
    "Zero-Shot Accuracy :",
    accuracy_score(y_true, y_pred)
)

print(
    "Zero-Shot Precision:",
    precision_score(y_true, y_pred)
)

print(
    "Zero-Shot Recall   :",
    recall_score(y_true, y_pred)
)

print(
    "Zero-Shot F1 Score :",
    f1_score(y_true, y_pred)
)

print(
    "Zero-Shot AUC      :",
    roc_auc_score(y_true, y_prob)
)

Zero-Shot Accuracy : 0.9185667752442996
Zero-Shot Precision: 0.8843469591226321
Zero-Shot Recall   : 0.9630836047774158
Zero-Shot F1 Score : 0.922037422037422
Zero-Shot AUC      : 0.952195189810443


-------------

### zero shot prediction for Simple R-GCN

In [45]:
print(type(node_embeddings))


<class 'torch.Tensor'>


In [46]:
print("predictor" in globals())

False


In [47]:
import os

print(os.listdir(project / "models"))

['DistMult', 'graphsage_model.pth', 'graphsage_predictor.pth', 'relation2id.pkl', 'rgcn_complete_checkpoint.pt', 'rgcn_model.pth', 'rgcn_node_embeddings.pt', 'rgcn_predictor.pth', 'TransE', 'tx_complete_checkpoint.pt', 'tx_model.pth', 'tx_predictor.pth']


### Load checkpoint

In [48]:
import torch

checkpoint = torch.load(
    project /
    "models" /
    "rgcn_complete_checkpoint.pt",
    map_location="cpu"
)

print(checkpoint.keys())

dict_keys(['model_state_dict', 'predictor_state_dict', 'node_embeddings', 'node2id', 'relation2id'])


### Recreate the Predictor

In [49]:
import torch.nn as nn

In [50]:
class LinkPredictor(nn.Module):

    def __init__(self, embedding_dim):

        super().__init__()

        self.linear = nn.Linear(
            embedding_dim * 2,
            1
        )

    def forward(self, source, target):

        x = torch.cat(
            [
                source,
                target
            ],
            dim=1
        )

        return self.linear(x)

### Restore the Trained Predictor

In [51]:
embedding_dim = checkpoint["node_embeddings"].shape[1]

predictor = LinkPredictor(
    embedding_dim
)

predictor.load_state_dict(
    checkpoint["predictor_state_dict"]
)

predictor.eval()

print("R-GCN predictor loaded successfully!")

R-GCN predictor loaded successfully!


### Run Zero-Shot Evaluation

In [52]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

predictor.eval()

with torch.no_grad():

    src_embed = node_embeddings[
        zs_all_src
    ]

    dst_embed = node_embeddings[
        zs_all_dst
    ]

    logits = predictor(
        src_embed,
        dst_embed
    )

    probs = torch.sigmoid(
        logits
    ).squeeze()

    preds = (
        probs > 0.5
    ).int()

y_true = zs_labels.numpy()
y_pred = preds.numpy()
y_prob = probs.numpy()

print("R-GCN Zero-Shot Accuracy :", accuracy_score(y_true, y_pred))
print("R-GCN Zero-Shot Precision:", precision_score(y_true, y_pred))
print("R-GCN Zero-Shot Recall   :", recall_score(y_true, y_pred))
print("R-GCN Zero-Shot F1 Score :", f1_score(y_true, y_pred))
print("R-GCN Zero-Shot AUC      :", roc_auc_score(y_true, y_prob))

R-GCN Zero-Shot Accuracy : 0.8984799131378935
R-GCN Zero-Shot Precision: 0.8760245901639344
R-GCN Zero-Shot Recall   : 0.9283387622149837
R-GCN Zero-Shot F1 Score : 0.9014232999472852
R-GCN Zero-Shot AUC      : 0.9467356564938503


-----------------

### For GraphSAGE

In [53]:
print(os.listdir(project / "models"))

['DistMult', 'graphsage_model.pth', 'graphsage_predictor.pth', 'relation2id.pkl', 'rgcn_complete_checkpoint.pt', 'rgcn_model.pth', 'rgcn_node_embeddings.pt', 'rgcn_predictor.pth', 'TransE', 'tx_complete_checkpoint.pt', 'tx_model.pth', 'tx_predictor.pth']


In [54]:
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv

### Define GraphSAGE

In [55]:
class GraphSAGE(nn.Module):

    def __init__(
        self,
        num_nodes,
        embedding_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        self.conv1 = SAGEConv(
            embedding_dim,
            128
        )

        self.conv2 = SAGEConv(
            128,
            128
        )

    def forward(
        self,
        edge_index
    ):

        x = self.embedding.weight

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        return x

### Link Predictor

In [56]:
class LinkPredictor(nn.Module):

    def __init__(
        self,
        embedding_dim
    ):

        super().__init__()

        self.linear = nn.Linear(
            embedding_dim * 2,
            1
        )

    def forward(
        self,
        source,
        target
    ):

        x = torch.cat(
            [
                source,
                target
            ],
            dim=1
        )

        return self.linear(
            x
        )

### Create the Model

In [57]:
graphsage_model = GraphSAGE(
    num_nodes=node_embeddings.shape[0],
    embedding_dim=128
)

graphsage_predictor = LinkPredictor(
    embedding_dim=128
)

### Load Saved Weights

In [58]:
graphsage_model.load_state_dict(
    torch.load(
        project /
        "models" /
        "graphsage_model.pth",
        map_location="cpu"
    )
)

graphsage_predictor.load_state_dict(
    torch.load(
        project /
        "models" /
        "graphsage_predictor.pth",
        map_location="cpu"
    )
)

graphsage_model.eval()
graphsage_predictor.eval()

print("GraphSAGE model loaded")

GraphSAGE model loaded


### Generate GraphSAGE Embeddings

In [59]:
with torch.no_grad():

    graphsage_embeddings = graphsage_model(
        edge_index
    )

print(graphsage_embeddings.shape)

torch.Size([7502, 128])


### Zero-Shot Evaluation

In [60]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

with torch.no_grad():

    src_embed = graphsage_embeddings[
        zs_all_src
    ]

    dst_embed = graphsage_embeddings[
        zs_all_dst
    ]

    logits = graphsage_predictor(
        src_embed,
        dst_embed
    )

    probs = torch.sigmoid(
        logits
    ).squeeze()

    preds = (
        probs > 0.5
    ).int()

y_true = zs_labels.numpy()
y_pred = preds.numpy()
y_prob = probs.numpy()

print("GraphSAGE Zero-Shot Accuracy :", accuracy_score(y_true, y_pred))
print("GraphSAGE Zero-Shot Precision:", precision_score(y_true, y_pred))
print("GraphSAGE Zero-Shot Recall   :", recall_score(y_true, y_pred))
print("GraphSAGE Zero-Shot F1 Score :", f1_score(y_true, y_pred))
print("GraphSAGE Zero-Shot AUC      :", roc_auc_score(y_true, y_prob))

GraphSAGE Zero-Shot Accuracy : 0.9071661237785016
GraphSAGE Zero-Shot Precision: 0.8478664192949907
GraphSAGE Zero-Shot Recall   : 0.992399565689468
GraphSAGE Zero-Shot F1 Score : 0.9144572286143071
GraphSAGE Zero-Shot AUC      : 0.924467220990261
